# CommerceIntel - Exploratory Data Analysis

This notebook runs EDA on cleaned Online Retail transaction data.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px
from src.config import PROCESSED_TRANSACTIONS_FILE
from src.analysis.eda import run_eda
from src.data.loader import load_raw_transactions
from src.data.cleaner import clean_transactions, save_cleaned_data

In [ ]:
if not PROCESSED_TRANSACTIONS_FILE.exists():
    raw = load_raw_transactions()
    clean = clean_transactions(raw)
    save_cleaned_data(clean)

transactions = pd.read_csv(PROCESSED_TRANSACTIONS_FILE, parse_dates=["invoice_date"])
transactions.head()

In [ ]:
summary = run_eda(transactions)
summary

In [ ]:
monthly = transactions.copy()
monthly["year_month"] = monthly["invoice_date"].dt.to_period("M").astype(str)
monthly_agg = monthly.groupby("year_month", as_index=False)["revenue"].sum()
px.line(monthly_agg, x="year_month", y="revenue", title="Monthly Revenue", markers=True)

In [ ]:
top_products = (
    transactions.groupby(["stock_code", "description"], as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
    .head(15)
)
px.bar(top_products, x="revenue", y="description", orientation="h", title="Top Products")

In [ ]:
clv = transactions.groupby("customer_id", as_index=False)["revenue"].sum()
px.histogram(clv, x="revenue", nbins=40, title="Customer Lifetime Value Distribution")